In [1]:
!pip install librosa soundfile tqdm

In [ ]:
# ============================================================
# STEP 4 : MFCC FEATURE EXTRACTION
# ============================================================

import os
import json
import librosa
import numpy as np
from tqdm import tqdm

# ============================================================
# PATHS
# ============================================================

base_dir = r"Enter the Address of the Parent folder containing all the folders"

audio_folder = os.path.join(base_dir, "segmented_audio")

output_X = os.path.join(base_dir, "X.npy")
output_y = os.path.join(base_dir, "y.npy")

label_map_file = os.path.join(base_dir, "label_map.json")
dataset_info_file = os.path.join(base_dir, "dataset_info.json")

# ============================================================
# CLASS LABELS
# ============================================================

classes = {
    "ANT": 0,
    "ANI": 1,
    "HUM": 2
}

# ============================================================
# MFCC PARAMETERS
# ============================================================

N_MFCC = 40

# ============================================================
# PASS 1
# FIND FRAME LENGTHS
# ============================================================

print("\nCalculating MFCC lengths...\n")

frame_lengths = []

class_counts = {}

for cls in classes.keys():

    folder = os.path.join(audio_folder, cls)

    # Get all WAV files
    wav_files = [f for f in os.listdir(folder) if f.endswith(".wav")]

    # Store the count
    class_counts[cls] = len(wav_files)

    # Process only WAV files
    for file in tqdm(wav_files, desc=cls):

        if file.endswith(".wav"):

            path = os.path.join(folder, file)

            y_audio, sr = librosa.load(
                path,
                sr=None
            )

            N_FFT = 2048
            HOP_LENGTH = 512
            WIN_LENGTH = 2048
            
            mfcc = librosa.feature.mfcc(
                y=audio,
                sr=sr,
                n_mfcc=N_MFCC,
                n_fft=N_FFT,
                hop_length=HOP_LENGTH,
                win_length=WIN_LENGTH
            )

            frame_lengths.append(mfcc.shape[1])

frame_lengths = np.array(frame_lengths)

print("\nTotal clips :", len(frame_lengths))
print("Minimum frames :", frame_lengths.min())
print("Maximum frames :", frame_lengths.max())
print("Average frames :", round(frame_lengths.mean(),2))

# ============================================================
# CHOOSE MAX LENGTH
# ============================================================

MAX_LEN = int(np.percentile(frame_lengths,95))

print("\nChosen max length (95th percentile):", MAX_LEN)

# ============================================================
# PASS 2
# CREATE DATASET
# ============================================================

print("\nExtracting MFCC features...\n")

X = []
y = []

for cls,label in classes.items():

    folder = os.path.join(audio_folder,cls)

    for file in tqdm(os.listdir(folder),desc=cls):

        if file.endswith(".wav"):

            path = os.path.join(folder,file)

            audio,sr = librosa.load(
                path,
                sr= None
            )

            N_FFT = 2048
            HOP_LENGTH = 512
            WIN_LENGTH = 2048
            
            mfcc = librosa.feature.mfcc(
                y=audio,
                sr=sr,
                n_mfcc=N_MFCC,
                n_fft=N_FFT,
                hop_length=HOP_LENGTH,
                win_length=WIN_LENGTH
            )

            mfcc = mfcc.T

            # Padding

            if mfcc.shape[0] < MAX_LEN:

                pad = MAX_LEN - mfcc.shape[0]

                mfcc = np.pad(
                    mfcc,
                    pad_width=((0,pad),(0,0)),
                    mode="constant"
                )

            # Truncation

            else:

                mfcc = mfcc[:MAX_LEN]

            X.append(mfcc)

            y.append(label)

# ============================================================
# CONVERT TO NUMPY
# ============================================================

X = np.array(X,dtype=np.float32)
y = np.array(y,dtype=np.int32)

# ============================================================
# SAVE
# ============================================================

np.save(output_X,X)
np.save(output_y,y)

with open(label_map_file,"w") as f:

    json.dump(classes,f,indent=4)

dataset_info = {

    "sample_rate": 40000,
    "n_mfcc": N_MFCC,
    "max_len": MAX_LEN,
    "total_samples": int(len(X)),
    "input_shape": list(X.shape),
    "class_counts": class_counts

}

with open(dataset_info_file,"w") as f:

    json.dump(dataset_info,f,indent=4)

# ============================================================
# SUMMARY
# ============================================================

print("\n===============================")
print("Feature Extraction Complete")
print("===============================")

print("X shape :",X.shape)
print("y shape :",y.shape)

print("\nSaved files:")

print(output_X)
print(output_y)
print(label_map_file)
print(dataset_info_file)

Task was destroyed but it is pending!
task: <Task pending name='Task-131' coro=<_async_in_context.<locals>.run_in_context_pre311() done, defined at C:\Users\gaura\AppData\Local\Programs\Python\Python310\lib\site-packages\ipykernel\utils.py:76> wait_for=<Task pending name='Task-132' coro=<_async_in_context.<locals>.preserve_context() running at C:\Users\gaura\AppData\Local\Programs\Python\Python310\lib\site-packages\ipykernel\utils.py:68> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\gaura\AppData\Local\Programs\Python\Python310\lib\site-packages\zmq\eventloop\zmqstream.py:563]>
C:\Users\gaura\AppData\Local\Programs\Python\Python310\lib\tokenize.py:527: RuntimeWarning: coroutine '_async_in_context.<locals>.preserve_context' was never awaited
  pseudomatch = _compile(PseudoToken).match(line, pos)
Task was destroyed but it is pending!
task: <Task pending name='Task-132' coro=<_async_in_context.<locals>.preserve_context() running at C:\Users\gaura\A


Calculating MFCC lengths...



ANT: 0it [00:00, ?it/s]
ANI: 0it [00:00, ?it/s]
HUM: 100%|██████████| 47/47 [00:00<00:00, 94.77it/s]



Total clips : 47
Minimum frames : 17
Maximum frames : 17
Average frames : 17.0

Chosen max length (95th percentile): 17

Extracting MFCC features...



ANT: 0it [00:00, ?it/s]
ANI: 0it [00:00, ?it/s]
HUM: 100%|██████████| 47/47 [00:00<00:00, 115.70it/s]


Feature Extraction Complete
X shape : (47, 17, 40)
y shape : (47,)

Saved files:
C:\Users\gaura\Downloads\IITG Internship\Annotated Data-4 (Gaurav)\All Annotated Data\Batch1\X.npy
C:\Users\gaura\Downloads\IITG Internship\Annotated Data-4 (Gaurav)\All Annotated Data\Batch1\y.npy
C:\Users\gaura\Downloads\IITG Internship\Annotated Data-4 (Gaurav)\All Annotated Data\Batch1\label_map.json
C:\Users\gaura\Downloads\IITG Internship\Annotated Data-4 (Gaurav)\All Annotated Data\Batch1\dataset_info.json
